# Pipeline GPU 2 : PyTorch -> torch.compile (Inductor) -> CUDA -> GPU
Runtime : GPU T4.

In [1]:
!nvidia-smi
!pip -q install nvidia-ml-py

Tue Nov  3 16:35:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
sys.path.append('/content/drive/MyDrive/tp_deploy')
from common import *
X, y = load_data()
print(X.shape, y[:10])

Mounted at /content/drive
torch.Size([50, 3, 224, 224]) [2 1 5 8 9 3 8 9 0 1]


In [3]:
model = load_model().cuda()
cmodel = torch.compile(model, backend='inductor')
t = time.perf_counter()
with torch.inference_mode():
    cmodel(X[:BS].cuda()); torch.cuda.synchronize()
print('première inférence (compilation incluse) :', time.perf_counter() - t, 's')

def predict(b):
    with torch.inference_mode():
        return cmodel(b.cuda()).cpu().numpy()

W1103 16:36:34.127000 480 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


première inférence (compilation incluse) : 21.680746759000044 s


In [4]:
res = run_benchmark('gpu_torchcompile', predict, X, y, device='gpu')

total_s      moy=0.1302 std=0.0011 min=0.1285 max=0.1312
throughput   moy=384.0894 std=3.1607 min=380.9550 max=389.1109
batch_ms     moy=26.0280 std=0.2118 min=25.6921 max=26.2397
top1         moy=0.9200 std=0.0000 min=0.9200 max=0.9200
énergie : {'watts': 67.63119592875319, 'joules_per_image': 0.17904232710331064}
